# Train YOLO on SoccerNet-v3 (Kaggle GPU)

## Setup

1. Enable **GPU** + **Internet**
2. **Add Input** → **Models** → your `best.pt` model
3. **Add Input** → **Datasets** → `soccernettoyolo` (contains `soccernet_to_yolo.py`)
4. Run all cells in order

In [ ]:
!pip install -q ultralytics SoccerNet tqdm

In [ ]:
import shutil
from pathlib import Path

from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames

SOCCERNET_DIR = Path("/kaggle/working/SoccerNet")
MAX_GAMES = 20  # ~2 GB zips; leaves room for YOLO images + training on 19.5 GiB

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))

games = getListGames("train", task="frames")[:MAX_GAMES]
print(f"Downloading {len(games)} games (~{MAX_GAMES * 80 // 1024} GB estimated)")

for i, game in enumerate(games, 1):
    print(f"[{i}/{len(games)}] {game}")
    downloader.downloadGame(
        game=game,
        files=["Frames-v3.zip", "Labels-v3.json"],
        spl="train",
    )

!df -h /kaggle/working
print("Download complete:", SOCCERNET_DIR)

In [ ]:
# Creates soccernet_to_yolo.py in /kaggle/working/ (no manual upload needed)
from pathlib import Path

Path("/kaggle/working/soccernet_to_yolo.py").write_text(r'''"""Convert SoccerNet-v3 JSON to Ultralytics YOLO format."""
from __future__ import annotations
import json, shutil, zipfile
from pathlib import Path
from tqdm import tqdm

YOLO_NAMES = ["ball", "player", "goalkeeper", "referee", "goalpost"]
SN_BBOX_CLASS_TO_YOLO = {
    "Ball": 0, "Player team left": 1, "Player team right": 1,
    "Player team unknown 1": 1, "Player team unknown 2": 1,
    "Goalkeeper team left": 2, "Goalkeeper team right": 2, "Goalkeeper team unknown": 2,
    "Main referee": 3, "Side referee": 3,
}
SN_GOAL_LINE_CLASS_TO_YOLO = {
    "Goal left post left ": 4, "Goal left post right": 4, "Goal left crossbar": 4,
    "Goal right post left": 4, "Goal right post right": 4, "Goal right crossbar": 4,
}

def _xyxy_to_yolo_line(cls_id, x1, y1, x2, y2, image_meta):
    w_img, h_img = float(image_meta["width"]), float(image_meta["height"])
    x_c = min(1.0, max(0.0, ((x1 + x2) / 2.0) / w_img))
    y_c = min(1.0, max(0.0, ((y1 + y2) / 2.0) / h_img))
    bw = min(1.0, max(0.0, abs(x2 - x1) / w_img))
    bh = min(1.0, max(0.0, abs(y2 - y1) / h_img))
    if bw <= 0 or bh <= 0: return None
    return f"{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}"

def bbox_to_yolo_line(bbox, image_meta):
    sn_class = bbox.get("class")
    if sn_class not in SN_BBOX_CLASS_TO_YOLO: return None
    p = bbox["points"]
    return _xyxy_to_yolo_line(SN_BBOX_CLASS_TO_YOLO[sn_class], float(p["x1"]), float(p["y1"]), float(p["x2"]), float(p["y2"]), image_meta)

def line_to_yolo_line(line, image_meta, padding=12.0):
    sn_class = line.get("class")
    if sn_class not in SN_GOAL_LINE_CLASS_TO_YOLO: return None
    points = line.get("points") or []
    if len(points) < 4: return None
    xs = [float(points[i]) for i in range(0, len(points), 2)]
    ys = [float(points[i]) for i in range(1, len(points), 2)]
    return _xyxy_to_yolo_line(SN_GOAL_LINE_CLASS_TO_YOLO[sn_class], min(xs)-padding, min(ys)-padding, max(xs)+padding, max(ys)+padding, image_meta)

def _extract_image(zip_path, image_name, dest_path):
    if dest_path.exists(): return True
    if not zip_path.exists(): return False
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        if image_name not in zf.namelist(): return False
        with zf.open(image_name) as src, open(dest_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return True

def convert_game(soccernet_root, game_rel_path, images_out, labels_out, stem_prefix):
    labels_path = soccernet_root / game_rel_path / "Labels-v3.json"
    if not labels_path.exists(): return 0
    metadata = json.loads(labels_path.read_text(encoding="utf-8"))
    zip_path = soccernet_root / metadata["GameMetadata"]["UrlLocal"] / "Frames-v3.zip"
    count = 0
    for action_name in metadata["GameMetadata"]["list_actions"]:
        img_names = [action_name] + metadata["actions"][action_name]["linked_replays"]
        for i, img_name in enumerate(img_names):
            img_type = "actions" if i == 0 else "replays"
            ann = metadata[img_type][img_name]
            yolo_lines = [l for b in ann.get("bboxes", []) if (l := bbox_to_yolo_line(b, ann["imageMetadata"]))]
            yolo_lines += [l for g in ann.get("lines", []) if (l := line_to_yolo_line(g, ann["imageMetadata"]))]
            lines = yolo_lines
            if not lines: continue
            safe_stem = f"{stem_prefix}_{img_name.replace('/', '_').replace('.png', '')}"
            image_out = images_out / f"{safe_stem}.png"
            label_out = labels_out / f"{safe_stem}.txt"
            if not _extract_image(zip_path, img_name, image_out): continue
            label_out.write_text("\\n".join(lines) + "\\n", encoding="utf-8")
            count += 1
    return count

def write_data_yaml(output_dir):
    yaml_path = output_dir / "data.yaml"
    yaml_path.write_text(f"path: {output_dir.resolve()}\\ntrain: images/train\\nval: images/val\\ntest: images/test\\nnc: {len(YOLO_NAMES)}\\nnames: {YOLO_NAMES}\\n", encoding="utf-8")
    return yaml_path

def convert_soccernet_v3(soccernet_root, output_dir, splits=None, max_games_per_split=None):
    from SoccerNet.utils import getListGames
    soccernet_root, output_dir = Path(soccernet_root), Path(output_dir)
    splits = splits or ["train", "valid", "test"]
    split_map = {"train": "train", "valid": "val", "test": "test"}
    total = 0
    for split in splits:
        images_out = output_dir / "images" / split_map.get(split, split)
        labels_out = output_dir / "labels" / split_map.get(split, split)
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)
        games = getListGames(split, task="frames")
        if max_games_per_split is not None: games = games[:max_games_per_split]
        for game in tqdm(games, desc=f"Converting {split}"):
            prefix = game.replace("/", "_").replace(" ", "_")
            total += convert_game(soccernet_root, game, images_out, labels_out, prefix)
    yaml_path = write_data_yaml(output_dir)
    print(f"Converted {total} images -> {output_dir}")
    return yaml_path
''', encoding="utf-8")
print("Wrote /kaggle/working/soccernet_to_yolo.py")

In [ ]:
import sys
from pathlib import Path

# Prefer the fixed converter written in the previous cell, then dataset input.
script = Path("/kaggle/working/soccernet_to_yolo.py")
if not script.exists():
    script = next(Path("/kaggle/input").rglob("soccernet_to_yolo.py"), None)
if script is None:
    raise FileNotFoundError("Run the converter cell above or add the soccernettoyolo dataset input.")
sys.path.insert(0, str(script.parent))
print("Using converter:", script)

from soccernet_to_yolo import convert_soccernet_v3

YOLO_DIR = "/kaggle/working/soccernet_yolo"
yaml_path = convert_soccernet_v3(
    soccernet_root=SOCCERNET_DIR,
    output_dir=YOLO_DIR,
    splits=["train"],
    max_games_per_split=MAX_GAMES,
)

# Free disk: delete zips after images are extracted (~2 GB back)
for zip_path in Path(SOCCERNET_DIR).rglob("Frames-v3.zip"):
    zip_path.unlink()
print("Deleted Frames-v3.zip files to save disk")

!df -h /kaggle/working
print("data.yaml:", yaml_path)

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

# Kaggle Model inputs mount under /kaggle/input/<your-model-slug>/
# Run `!ls /kaggle/input` in a cell if you need to confirm the path.
WEIGHTS_IN = next(Path("/kaggle/input").rglob("best.pt"), None)

if WEIGHTS_IN and WEIGHTS_IN.exists():
    shutil.copy(WEIGHTS_IN, "/kaggle/working/best.pt")
    MODEL = "/kaggle/working/best.pt"
    print("Fine-tuning from Kaggle model:", WEIGHTS_IN)
else:
    MODEL = "yolov8m.pt"
    print("No best.pt in /kaggle/input — training from", MODEL)

model = YOLO(MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="/kaggle/working/runs",
    name="soccernet_v3",
    device=0,
    lr0=1e-4,
    mosaic=1.0,
    mixup=0.1,
)
print("Best:", results.save_dir / "weights" / "best.pt")

In [ ]:
from pathlib import Path
from IPython.display import FileLink
from ultralytics import YOLO

# Training may save to soccernet_v3, soccernet_v3-2, etc. if you re-ran the cell
best = next(Path("/kaggle/working/runs").rglob("best.pt"), None)
if best is None:
    raise FileNotFoundError("No best.pt found under /kaggle/working/runs")

print("Best weights:", best)
YOLO(str(best)).val(data=str(yaml_path))
FileLink(str(best))